# Diffusion Model for Financial Time Series

**Before running:** Runtime → Change runtime type → Hardware accelerator → **GPU**

| Cell | What it does |
|------|--------------|
| Setup | Mount Drive, set paths, check GPU |
| Config | All hyperparameters in one place |
| Train | Train the diffusion model |
| Sample | Generate synthetic time series from a checkpoint |
| Evaluate | Compute discriminative/predictive/VDS/FDDS scores |

## 1. Setup

In [1]:
# ── Check GPU ────────────────────────────────────────────────────────────────
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory  : {mem_gb:.1f} GB")

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : NVIDIA A100-SXM4-40GB
Memory  : 42.4 GB


In [2]:
# ── Clone repo from GitHub ────────────────────────────────────────────────────
import os
REPO_DIR = '/content/DiffusionModelTimeSeries'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Ardameliksah/DiffusionModelTimeSeries.git {REPO_DIR}
else:
    print("Repo already cloned, pulling latest...")
    !git -C {REPO_DIR} pull

Repo already cloned, pulling latest...
Already up to date.


In [3]:
import os, sys
from pathlib import Path

REPO_PATH = REPO_DIR  # set by the clone cell above

assert Path(REPO_PATH).exists(), f"Folder not found: {REPO_PATH}"

os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print(f"Working directory: {os.getcwd()}")
print("Files found:", [f for f in os.listdir() if f.endswith('.py')])

Working directory: /content/DiffusionModelTimeSeries
Files found: ['evaluate_unified.py', 'visualize_samples.py', 'sanity.py', 'quick_range_check.py', 'sample_unified.py', '__init__.py', 'eval_metrics.py', 'input_test.py', 'train_with_mode.py', 'diagnose_x0_range.py', 'diag_t_bins.py']


In [4]:
# ── Install any missing packages ────────────────────────────────────────────
# Colab already has torch, numpy, pandas, matplotlib, scikit-learn, scipy,
# seaborn, tqdm, pillow. Only install what might be missing.
!pip install -q --upgrade pip
!pip install -q wandb
!pip show tqdm scikit-learn seaborn | grep -E 'Name|Version'

Name: tqdm
Version: 4.67.3
Name: scikit-learn
Version: 1.6.1
 Name: GCC runtime library
 Version 3.1, 31 March 2009
Name: seaborn
Version: 0.13.2


In [9]:
# ── Weights & Biases login ────────────────────────────────────────────────────
# Option A: set a Colab Secret named WANDB_API_KEY (Secrets panel, left sidebar)
# Option B: enter your key interactively when prompted below
import wandb, os
#wandb apı key:
os.environ["WANDB_API_KEY"] = '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'

_api_key = os.environ.get("WANDB_API_KEY")   # populated by Colab Secrets
if _api_key:
    wandb.login(key=_api_key, relogin=False)
else:
    wandb.login()                              # interactive prompt — paste API key

print("wandb version:", wandb.__version__)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: a-meliksahdemir (a-meliksahdemir-bo-azi-i-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb version: 0.26.1


## 2. Overnight Experiment Loop

Edit `EXPERIMENTS` below to add/remove/reorder runs. Each experiment logs to W&B automatically.  
All results land in separate `output/ckpt_<name>/` folders so nothing overwrites anything else.

In [ ]:
import torch
import gc
import importlib
import train_with_mode as _twm
importlib.reload(_twm)          # always use the latest code after a git pull
from train_with_mode import train

# ── Shared settings ────────────────────────────────────────────────────────────
DEVICE             = "cuda" if torch.cuda.is_available() else "cpu"
WANDB_PROJECT      = "diffusion-timeseries"
NUM_WORKERS        = 2
SEED               = 42
BATCH_SIZE         = 64
IMG_PRED_OBJECTIVE = "pred_x0"
IMG_LOSS_TYPE      = "l1"

# ── Inline metric settings ─────────────────────────────────────────────────────
# These run DURING training every EVAL_EVERY epochs.
# disc_iterations=500 / pred_iterations=1000 (fast) are used for W&B tracking.
# Full evaluation (disc=2000, pred=5000) is run separately after training via evaluate_unified.py.
EVAL_METRICS_EVERY  = 200   # every 200 epochs → 5 evals per 1000-epoch run (~30s each)
N_METRIC_ITERATIONS = 1     # 1 run per eval (fast; full eval does 3–5 runs)
NUM_METRIC_SAMPLES  = 128   # samples to compare real vs fake

# ── Experiment list ────────────────────────────────────────────────────────────
# Architecture key:
#   mode="raw"           → Transformer (base L1 loss only)
#   mode="decomposition" → Transformer (L1 + optional FFT / trend / season loss)
#   mode="image"         → UNet on 2D image representation of time series
#
# Runs are ordered: RAW baseline → DECOMPOSITION variants → IMAGE (UNet) small

EXPERIMENTS = [

    # ══════════════════════════════════════════════════════════════════════════
    # TRANSFORMER — RAW baseline  (no extra loss terms)
    # ══════════════════════════════════════════════════════════════════════════
    dict(name="raw_medium_h128_l6",
         mode="raw",
         hidden_dim=128, num_layers=6, num_epochs=1000, lr=1e-4,
         fft_weight=0.0, trend_weight=0.0, season_weight=0.0),

    # ══════════════════════════════════════════════════════════════════════════
    # TRANSFORMER — DECOMPOSITION  small  (h64 / l3)
    # ══════════════════════════════════════════════════════════════════════════
    dict(name="decomp_small_fft_only",
         mode="decomposition",
         hidden_dim=64, num_layers=3, num_epochs=1000, lr=1e-4,
         fft_weight=0.1, trend_weight=0.0, season_weight=0.0),

    dict(name="decomp_small_trend_only",
         mode="decomposition",
         hidden_dim=64, num_layers=3, num_epochs=1000, lr=1e-4,
         fft_weight=0.0, trend_weight=0.5, season_weight=0.0),

    dict(name="decomp_small_season_only",
         mode="decomposition",
         hidden_dim=64, num_layers=3, num_epochs=1000, lr=1e-4,
         fft_weight=0.0, trend_weight=0.0, season_weight=0.5),

    dict(name="decomp_small_trend_season",
         mode="decomposition",
         hidden_dim=64, num_layers=3, num_epochs=1000, lr=1e-4,
         fft_weight=0.0, trend_weight=0.5, season_weight=0.5),

    # ══════════════════════════════════════════════════════════════════════════
    # TRANSFORMER — DECOMPOSITION  medium  (h128 / l6)
    # ══════════════════════════════════════════════════════════════════════════
    dict(name="decomp_medium_fft_only",
         mode="decomposition",
         hidden_dim=128, num_layers=6, num_epochs=1000, lr=1e-4,
         fft_weight=0.1, trend_weight=0.0, season_weight=0.0),

    dict(name="decomp_medium_trend_only",
         mode="decomposition",
         hidden_dim=128, num_layers=6, num_epochs=1000, lr=1e-4,
         fft_weight=0.0, trend_weight=0.5, season_weight=0.0),

    dict(name="decomp_medium_season_only",
         mode="decomposition",
         hidden_dim=128, num_layers=6, num_epochs=1000, lr=1e-4,
         fft_weight=0.0, trend_weight=0.0, season_weight=0.5),

    dict(name="decomp_medium_trend_season",
         mode="decomposition",
         hidden_dim=128, num_layers=6, num_epochs=1000, lr=1e-4,
         fft_weight=0.0, trend_weight=0.5, season_weight=0.5),

    # ══════════════════════════════════════════════════════════════════════════
    # TRANSFORMER — DECOMPOSITION  large  (h256 / l8)
    # ══════════════════════════════════════════════════════════════════════════
    dict(name="decomp_large_fft_only",
         mode="decomposition",
         hidden_dim=256, num_layers=8, num_epochs=1000, lr=1e-4,
         fft_weight=0.1, trend_weight=0.0, season_weight=0.0),

    dict(name="decomp_large_trend_only",
         mode="decomposition",
         hidden_dim=256, num_layers=8, num_epochs=1000, lr=1e-4,
         fft_weight=0.0, trend_weight=0.5, season_weight=0.0),

    dict(name="decomp_large_season_only",
         mode="decomposition",
         hidden_dim=256, num_layers=8, num_epochs=1000, lr=1e-4,
         fft_weight=0.0, trend_weight=0.0, season_weight=0.5),

    dict(name="decomp_large_trend_season",
         mode="decomposition",
         hidden_dim=256, num_layers=8, num_epochs=1000, lr=1e-4,
         fft_weight=0.0, trend_weight=0.5, season_weight=0.5),

    # ══════════════════════════════════════════════════════════════════════════
    # UNET — IMAGE mode  small only  (mc64 / nb2)
    #   delay  → wraps time series into 8×8 delay-coordinate matrix
    #   stft   → converts to spectrogram (freq × time image)
    # ══════════════════════════════════════════════════════════════════════════
    dict(name="image_delay_unet_mc64_nb2",
         mode="image", embedding="delay",
         hidden_dim=64, num_layers=2, num_epochs=500, lr=1e-4,
         fft_weight=None, trend_weight=None, season_weight=None),

    dict(name="image_stft_unet_mc64_nb2",
         mode="image", embedding="stft",
         hidden_dim=64, num_layers=2, num_epochs=500, lr=1e-4,
         fft_weight=None, trend_weight=None, season_weight=None),
]

# ── Run all experiments sequentially ──────────────────────────────────────────
print(f"Device : {DEVICE}")
print(f"Total  : {len(EXPERIMENTS)} experiments\n")
print("  #   Name                              Mode          Architecture")
print("  " + "-"*65)
_arch = {"raw": "Transformer", "decomposition": "Transformer", "image": "UNet"}
for i, exp in enumerate(EXPERIMENTS):
    emb = f" [{exp.get('embedding','—')}]" if exp["mode"] == "image" else ""
    print(f"  {i+1:2d}. {exp['name']:<36} {exp['mode']:<14} {_arch[exp['mode']]}{emb}")
print()

for i, exp in enumerate(EXPERIMENTS):
    print(f"\n{'='*70}")
    print(f"  [{i+1}/{len(EXPERIMENTS)}]  {exp['name']}")
    print(f"{'='*70}\n")
    try:
        train(
            mode               = exp["mode"],
            device             = DEVICE,
            embedding          = exp.get("embedding", "delay"),
            hidden_dim         = exp["hidden_dim"],
            num_layers         = exp["num_layers"],
            num_epochs         = exp["num_epochs"],
            lr                 = exp["lr"],
            batch_size         = BATCH_SIZE,
            num_workers        = NUM_WORKERS,
            seed               = SEED,
            use_wandb          = True,
            wandb_project      = WANDB_PROJECT,
            wandb_run_name     = exp["name"],
            eval_metrics       = True,
            eval_metrics_every = EVAL_METRICS_EVERY,
            n_metric_iterations= N_METRIC_ITERATIONS,
            num_metric_samples = NUM_METRIC_SAMPLES,
            img_pred_objective = IMG_PRED_OBJECTIVE,
            img_loss_type      = IMG_LOSS_TYPE,
            fft_weight         = exp.get("fft_weight"),
            trend_weight       = exp.get("trend_weight"),
            season_weight      = exp.get("season_weight"),
            checkpoint_dir     = f"output/ckpt_{exp['name']}",
        )
    except Exception as e:
        import traceback
        print(f"\n!!! EXPERIMENT FAILED: {exp['name']}")
        traceback.print_exc()

    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "="*70)
print("  ALL EXPERIMENTS COMPLETE")
print("="*70)

## 3. (Optional) Single Manual Run

Use this to run one specific config manually after the overnight loop, or for quick tests.

In [ ]:
from train_with_mode import train
import torch

# ── Single-run config — edit and run this cell manually ───────────────────────
MODE      = "image"
EMBEDDING = "delay"
HIDDEN_DIM     = 128
NUM_LAYERS     = 6
NUM_EPOCHS     = 500
LEARNING_RATE  = 1e-4
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

train(
    mode               = MODE,
    device             = DEVICE,
    embedding          = EMBEDDING,
    hidden_dim         = HIDDEN_DIM,
    num_layers         = NUM_LAYERS,
    num_epochs         = NUM_EPOCHS,
    lr                 = LEARNING_RATE,
    batch_size         = 64,
    num_workers        = 2,
    seed               = 42,
    use_wandb          = True,
    wandb_project      = "diffusion-timeseries",
    eval_metrics       = True,
    eval_metrics_every = 100,
    n_metric_iterations= 3,
    num_metric_samples = 128,
    img_pred_objective = "pred_x0",
    img_loss_type      = "l1",
)

## 4. Sample

Generates `NUM_SAMPLES` synthetic windows from a trained checkpoint and saves them as `.npz` to `output/generated_samples/`.

In [ ]:
from pathlib import Path
from sample_unified import generate, denormalize, save_samples
from config.stocks_config import Config as RawConfig
from config.image_config import ImageVersionConfig
from utils.data_utils import StockDataset, build_scaler

# Auto-detect best_model.pt if not set manually
_ckpt = CHECKPOINT_PATH
if _ckpt is None:
    if MODE == "raw":
        _ckpt = str(Path(REPO_PATH) / "output" / "checkpoints" / "best_model.pt")
    else:
        _ckpt = str(Path(REPO_PATH) / "output" / "checkpoints_image" / "best_model.pt")

assert Path(_ckpt).exists(), f"Checkpoint not found: {_ckpt}"
print(f"Loading checkpoint: {_ckpt}")

# Build config
if MODE == "raw":
    _config = RawConfig()
else:
    _config = ImageVersionConfig()
    _config.image.embedding_type = EMBEDDING

if NORMALIZATION is not None:
    _config.data.neg_one_to_one = (NORMALIZATION == "minmax")

# Generate
samples = generate(
    checkpoint_path=_ckpt,
    mode=MODE,
    config=_config,
    num_samples=NUM_SAMPLES,
    num_steps=NUM_STEPS,
    eta=ETA,
    device=DEVICE,
)

# Denormalize back to original price scale
dataset = StockDataset(
    csv_path=_config.data.data_path,
    scaler=build_scaler(_config.data.neg_one_to_one),
    window_length=_config.model.sequence_length,
)
samples_denorm = denormalize(samples, dataset)

_emb = EMBEDDING if MODE == "image" else None
save_samples(samples_denorm, _config.sampling.output_dir, MODE, _emb)
print(f"\nFinal shape: {samples_denorm.shape}  (samples, channels, time)")

### Quick visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_show = min(6, samples_denorm.shape[0])
channel_names = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]

fig, axes = plt.subplots(n_show, 1, figsize=(12, 2 * n_show))
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    # Plot Close price (channel 3) for readability
    ax.plot(samples_denorm[i, 3, :], linewidth=1)
    ax.set_ylabel("Close", fontsize=8)
    ax.set_title(f"Sample {i+1}", fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle(f"Generated Samples — {MODE} mode", fontsize=11)
plt.tight_layout()
plt.show()

## 5. Evaluate

Computes:
- **Discriminative score** — GRU classifier real vs synthetic (target: 0.0, test acc: 0.5)
- **Predictive MAE** — train-on-fake / test-on-real (lower = better)
- **VDS** — KL divergence of value distributions (lower = better)
- **FDDS** — KL divergence of cross-correlation distributions (lower = better)
- **Correlational score** — |CACF_fake − CACF_real| / 10 (lower = better)

Results and a comparison plot are saved to `output/` automatically.

In [ ]:
from evaluate_unified import evaluate

evaluate(
    mode=MODE,
    checkpoint_path=_ckpt,
    device=DEVICE,
    num_samples=NUM_SAMPLES,
    output_dir=None,               # None = config default
    n_metric_iterations=N_METRIC_ITERATIONS,
    compute_context_fid=COMPUTE_CONTEXT_FID,
    normalization=NORMALIZATION,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    seed=SEED,
    pos_enc=POS_ENC,
    use_wandb=USE_WANDB,           # ← logs all eval metrics to W&B
    wandb_project=WANDB_PROJECT,
)

## (Optional) Copy outputs back to Drive

Colab runtimes are ephemeral — outputs written inside `/content/` are lost when the session ends.  
If your `REPO_PATH` already points to Drive, checkpoints are already persistent. Otherwise run the cell below.

In [ ]:
# Only needed if REPO_PATH is NOT on Drive (e.g. you cloned to /content/)
import shutil

DRIVE_BACKUP = '/content/drive/MyDrive/TezBaselines/MyCode/output'
LOCAL_OUTPUT = str(Path(REPO_PATH) / 'output')

shutil.copytree(LOCAL_OUTPUT, DRIVE_BACKUP, dirs_exist_ok=True)
print(f"Outputs copied to {DRIVE_BACKUP}")